# RQ1: Global Electricity Sufficiency, 2015

This publication-ready notebook reproduces the core RQ1 analytical tables and figures from the cleaned GDL-level electricity sufficiency dataset. It combines the mapping/descriptive workflow and the sufficiency-gap inequality workflow into one notebook.

Core outputs are written to `Results/RQ1/`; intermediate diagnostics and method metadata are written to `Process/`.

## 1. Setup

Define input data, thresholds, output folders, plotting defaults, and reusable colors. The notebook starts from the validated GDL-level sufficiency table (`Data/global_SE_EC_sufficiency_2015.csv`). Pixel-level raster aggregation inputs and indicator definitions are documented in `Process/rq1_sufficiency_indicator_metadata_2015.json`.

In [ ]:

from pathlib import Path
import json
import os
import shutil
import warnings

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib-energy-sufficiency")

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
mpl.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize, LogNorm, TwoSlopeNorm
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Patch
import seaborn as sns

warnings.filterwarnings("ignore", category=UserWarning)

BASE_DIR = Path("/Users/nico/Research/Energy sufficiency")
DATA_PATH = BASE_DIR / "Data" / "global_SE_EC_sufficiency_2015.csv"
SHP_PATH = Path("/Users/nico/Documents/PhD research/energy poverty/Data/GDL Shapefiles V4/GDL Shapefiles V4.shp")
PROCESS_DIR = BASE_DIR / "Process"
RESULTS_DIR = BASE_DIR / "Results" / "RQ1"
FIG_DIR = RESULTS_DIR / "Figures"
MAP_DIR = FIG_DIR / "Maps"
STAT_DIR = FIG_DIR / "Statistics"
INEQ_DIR = FIG_DIR / "Inequality"
TABLE_DIR = RESULTS_DIR / "Tables"

for d in [PROCESS_DIR, RESULTS_DIR, FIG_DIR, MAP_DIR, STAT_DIR, INEQ_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Remove old combined figures and old root-level PNGs, keeping the output folder clean.
for pattern in ["*.png", "*.pdf"]:
    for p in FIG_DIR.glob(pattern):
        p.unlink()
for folder in [MAP_DIR, STAT_DIR, INEQ_DIR]:
    for p in folder.glob("*.png"):
        p.unlink()
    for p in folder.glob("*.pdf"):
        p.unlink()

FLOOR_THRESHOLD = 300.0
UPPER_THRESHOLD = 5000.0

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.titlesize": 12,
    "axes.labelsize": 9.5,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "legend.fontsize": 8,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.edgecolor": "#222222",
    "axes.linewidth": 0.9,
})
sns.set_theme(style="whitegrid", rc={"font.family": "DejaVu Sans"})


## 2. Region System and Palette

The seven analytical regions are kept stable across all RQ1 figures. The palette follows the agreed color scheme and is saved as a reference table for later RQs.

In [ ]:

REFERENCE_COLORS = {
    "light_blue": "#afd3e7",
    "blue": "#53a8e1",
    "light_red": "#fca3a3",
    "red": "#ed5f5f",
    "light_orange": "#fdcd8c",
    "orange": "#ffa74f",
    "lavender": "#d0bbdb",
    "purple": "#9a72c7",
}

REGION_ORDER = [
    "Sub-Saharan Africa",
    "Northern Africa and Western Asia",
    "South Asia",
    "Eastern and South-eastern Asia",
    "Europe",
    "Latin America and the Caribbean",
    "Northern America and Oceania",
]

REGION_COLORS = {
    "Sub-Saharan Africa": REFERENCE_COLORS["blue"],
    "Northern Africa and Western Asia": REFERENCE_COLORS["orange"],
    "South Asia": REFERENCE_COLORS["red"],
    "Eastern and South-eastern Asia": REFERENCE_COLORS["purple"],
    "Europe": REFERENCE_COLORS["light_blue"],
    "Latin America and the Caribbean": REFERENCE_COLORS["light_red"],
    "Northern America and Oceania": REFERENCE_COLORS["light_orange"],
}

SHORT_REGION_LABELS = [
    "Sub-Saharan\nAfrica",
    "N. Africa\n& W. Asia",
    "South\nAsia",
    "E. & SE\nAsia",
    "Europe",
    "Latin America\n& Caribbean",
    "N. America\n& Oceania",
]

region_palette = pd.DataFrame({
    "analytical_region": REGION_ORDER,
    "short_label": SHORT_REGION_LABELS,
    "color_hex": [REGION_COLORS[r] for r in REGION_ORDER],
})
region_palette.to_csv(TABLE_DIR / "rq1_region_palette.csv", index=False)
region_palette.to_csv(PROCESS_DIR / "rq1_region_palette.csv", index=False)
region_palette


In [ ]:

SOUTH_ASIA = {"AFG", "BGD", "BTN", "IND", "IRN", "LKA", "MDV", "NPL", "PAK"}
EAST_SE_ASIA = {"BRN", "CHN", "IDN", "JPN", "KHM", "KOR", "LAO", "MMR", "MNG", "MYS", "PHL", "PRK", "SGP", "THA", "TLS", "VNM"}
NORTH_AFRICA_WEST_ASIA = {"DZA", "BHR", "EGY", "IRQ", "ISR", "JOR", "KWT", "LBN", "LBY", "MAR", "OMN", "PSE", "QAT", "SAU", "SYR", "TUN", "TUR", "ARE", "YEM", "ARM", "AZE", "CYP", "GEO", "KAZ", "KGZ", "TJK", "TKM", "UZB"}
LATIN_AMERICA_CARIBBEAN = {"ATG", "ARG", "BHS", "BRB", "BLZ", "BOL", "BRA", "CHL", "COL", "CRI", "CUB", "DMA", "DOM", "ECU", "SLV", "GRD", "GTM", "GUY", "HTI", "HND", "JAM", "MEX", "NIC", "PAN", "PRY", "PER", "KNA", "LCA", "VCT", "SUR", "TTO", "URY", "VEN"}
NORTH_AMERICA_OCEANIA = {"AUS", "CAN", "FSM", "FJI", "KIR", "MHL", "NRU", "NZL", "PLW", "PNG", "SLB", "TON", "TUV", "USA", "VUT", "WSM"}
EUROPE = {"ALB", "AND", "AUT", "BEL", "BGR", "BIH", "BLR", "CHE", "CZE", "DEU", "DNK", "ESP", "EST", "FIN", "FRA", "GBR", "GRC", "HRV", "HUN", "IRL", "ISL", "ITA", "LTU", "LUX", "LVA", "MDA", "MKD", "MLT", "MNE", "NLD", "NOR", "POL", "PRT", "ROU", "RUS", "SRB", "SVK", "SVN", "SWE", "UKR"}

def assign_analytical_region(iso_code):
    if pd.isna(iso_code):
        return np.nan
    iso = str(iso_code).upper()
    if iso in SOUTH_ASIA:
        return "South Asia"
    if iso in EAST_SE_ASIA:
        return "Eastern and South-eastern Asia"
    if iso in NORTH_AFRICA_WEST_ASIA:
        return "Northern Africa and Western Asia"
    if iso in LATIN_AMERICA_CARIBBEAN:
        return "Latin America and the Caribbean"
    if iso in NORTH_AMERICA_OCEANIA:
        return "Northern America and Oceania"
    if iso in EUROPE:
        return "Europe"
    return "Sub-Saharan Africa"


## 3. Load and Validate Core Sufficiency Data

The four sufficiency indicators are already included in the clean GDL table. Definitions from the indicator metadata are:

- `sufficiency_ratio`: GDL-level `EC2015_pc / 300`.
- `sufficiency_gap`: GHSL-population-weighted share of mapped 1 km electricity pixels within each GDL below 300 kWh/cap/year.
- `sufficiency_deficit`: GHSL-population-weighted mean `max(0, 1 - pixel electricity_pc / 300)`.
- `sufficiency_excess`: GHSL-population-weighted share of mapped pixels above 5,000 kWh/cap/year.

Rows with invalid population or electricity were set to NA during indicator construction and are not forced back into the analysis.

In [ ]:

df = pd.read_csv(DATA_PATH)
metric_cols = ["EC2015_pc", "sufficiency_ratio", "sufficiency_gap", "sufficiency_deficit", "sufficiency_excess"]
indicator_cols = ["sufficiency_ratio", "sufficiency_gap", "sufficiency_deficit", "sufficiency_excess"]
for col in ["population", "EC2015_sum"] + metric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)

df["iso_code_for_region"] = df["iso_code"]
missing_iso = df["iso_code_for_region"].isna() & df["GDLcode"].notna()
df.loc[missing_iso, "iso_code_for_region"] = df.loc[missing_iso, "GDLcode"].astype(str).str[:3].str.upper()
df["analytical_region"] = df["iso_code_for_region"].apply(assign_analytical_region)
df["analytical_region"] = pd.Categorical(df["analytical_region"], categories=REGION_ORDER, ordered=True)

gdf = gpd.read_file(SHP_PATH)[["GDLcode", "geometry"]]
gdf = gdf.merge(df, on="GDLcode", how="right")
gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs="EPSG:4326")
gdf["geometry"] = gdf.geometry.simplify(0.06, preserve_topology=True)
gdf["log10_sufficiency_ratio_for_map"] = np.nan
mask_ratio_map = gdf["sufficiency_ratio"] > 0
gdf.loc[mask_ratio_map, "log10_sufficiency_ratio_for_map"] = np.log10(gdf.loc[mask_ratio_map, "sufficiency_ratio"])

print("Rows:", len(gdf))
print("Rows with analytical region:", int(df["analytical_region"].notna().sum()))
print("Rows with all four indicators:", int(gdf[indicator_cols].notna().all(axis=1).sum()))
print(gdf[metric_cols].isna().sum())


## 4. Regional and Country Summary Tables

Create population-weighted descriptive summaries for the full RQ1 table. These are the tabular basis for regional composition, breadth-depth, and summary heatmap figures.

In [ ]:

def weighted_mean(x, w):
    x = pd.to_numeric(x, errors="coerce")
    w = pd.to_numeric(w, errors="coerce")
    mask = x.notna() & w.notna() & (w > 0)
    if mask.sum() == 0:
        return np.nan
    return float(np.average(x[mask], weights=w[mask]))

def weighted_quantile(values, weights, qs):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    values, weights = values[mask], weights[mask]
    if len(values) == 0:
        return [np.nan] * len(qs)
    order = np.argsort(values)
    values, weights = values[order], weights[order]
    cumw = np.cumsum(weights) / np.sum(weights)
    return [float(np.interp(q, cumw, values)) for q in qs]

summary_rows = []
for region in REGION_ORDER:
    sub = df[df["analytical_region"] == region].copy()
    row = {
        "analytical_region": region,
        "n_gdl_total": int(len(sub)),
        "n_gdl_valid": int(sub[indicator_cols].notna().all(axis=1).sum()),
        "population_sum": float(sub.loc[sub["population"] > 0, "population"].sum()),
    }
    for col in metric_cols:
        x = sub[col]
        w = sub["population"]
        row[f"{col}_mean"] = float(x.mean(skipna=True))
        row[f"{col}_median"] = float(x.median(skipna=True))
        row[f"{col}_p25"] = float(x.quantile(0.25))
        row[f"{col}_p75"] = float(x.quantile(0.75))
        row[f"{col}_pop_weighted_mean"] = weighted_mean(x, w)
        q10, q50, q90 = weighted_quantile(x, w, [0.10, 0.50, 0.90])
        row[f"{col}_pop_weighted_p10"] = q10
        row[f"{col}_pop_weighted_p50"] = q50
        row[f"{col}_pop_weighted_p90"] = q90
    summary_rows.append(row)

region_summary = pd.DataFrame(summary_rows)
region_summary.to_csv(TABLE_DIR / "rq1_regional_sufficiency_summary.csv", index=False)

country_rows = []
for (iso, country, region), sub in df.groupby(["iso_code", "country", "analytical_region"], observed=True, dropna=False):
    row = {
        "iso_code": iso,
        "country": country,
        "analytical_region": region,
        "n_gdl_total": int(len(sub)),
        "n_gdl_valid": int(sub[indicator_cols].notna().all(axis=1).sum()),
        "population_sum": float(sub.loc[sub["population"] > 0, "population"].sum()),
    }
    for col in metric_cols:
        row[f"{col}_pop_weighted_mean"] = weighted_mean(sub[col], sub["population"])
        row[f"{col}_median"] = float(sub[col].median(skipna=True))
    country_rows.append(row)
country_summary = pd.DataFrame(country_rows).sort_values(["analytical_region", "country"])
country_summary.to_csv(TABLE_DIR / "rq1_country_sufficiency_summary.csv", index=False)
region_summary.head()


## 5. Global Maps

Generate single-map PNGs for electricity consumption and each sufficiency indicator. `No data` is shown separately in grey. `sufficiency_ratio` uses a diverging log-scale palette centered on ratio = 1, because values below, equal to, and above 1 have different interpretations.

In [ ]:

map_specs = {
    "EC2015_pc": {
        "title": "Electricity consumption",
        "cmap": LinearSegmentedColormap.from_list("ec_teal", ["#f3fbfb", "#afd3e7", "#53a8e1", "#2b5c9c"]),
        "norm": LogNorm(vmin=1, vmax=50000),
        "ticks": [1, 100, 1000, 10000, 50000],
        "ticklabels": ["1", "100", "1k", "10k", "50k"],
        "legend_label": "kWh/cap/year",
        "filename": "Map_electricity_consumption.png",
    },
    "sufficiency_ratio": {
        "title": "Sufficiency ratio",
        "plot_column": "log10_sufficiency_ratio_for_map",
        "cmap": LinearSegmentedColormap.from_list("ratio_diverging", ["#b2182b", "#fca3a3", "#f7f7f7", "#afd3e7", "#2166ac"]),
        "norm": TwoSlopeNorm(vmin=-3, vcenter=0, vmax=2),
        "ticks": [-3, -1, 0, 1, 2],
        "ticklabels": ["0.001", "0.1", "1", "10", "100"],
        "legend_label": "Ratio",
        "filename": "Map_sufficiency_ratio.png",
    },
    "sufficiency_gap": {
        "title": "Sufficiency gap",
        "cmap": LinearSegmentedColormap.from_list("gap_red", ["#fff7f7", "#fca3a3", "#ed5f5f", "#9b1d27"]),
        "norm": Normalize(vmin=0, vmax=1),
        "ticks": [0, 0.5, 1],
        "ticklabels": ["0", "0.5", "1"],
        "legend_label": "Share",
        "filename": "Map_sufficiency_gap.png",
    },
    "sufficiency_deficit": {
        "title": "Sufficiency deficit",
        "cmap": LinearSegmentedColormap.from_list("deficit_orange", ["#fff9ed", "#fdcd8c", "#ffa74f", "#b85d00"]),
        "norm": Normalize(vmin=0, vmax=1),
        "ticks": [0, 0.5, 1],
        "ticklabels": ["0", "0.5", "1"],
        "legend_label": "Index",
        "filename": "Map_sufficiency_deficit.png",
    },
    "sufficiency_excess": {
        "title": "Sufficiency excess",
        "cmap": LinearSegmentedColormap.from_list("excess_purple", ["#fbf8fd", "#d0bbdb", "#9a72c7", "#5d3a91"]),
        "norm": Normalize(vmin=0, vmax=1),
        "ticks": [0, 0.5, 1],
        "ticklabels": ["0", "0.5", "1"],
        "legend_label": "Share",
        "filename": "Map_sufficiency_excess.png",
    },
}

# Map plotting with visible border and short tick marks like the reference style.
def plot_world_map(column):
    spec = map_specs[column]
    plot_column = spec.get("plot_column", column)

    fig = plt.figure(figsize=(12.5, 5.8))
    gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[17, 2.7], hspace=0.42)
    ax = fig.add_subplot(gs[0])
    legend_ax = fig.add_subplot(gs[1])
    legend_ax.axis("off")

    gdf.plot(ax=ax, color="#d9d9d9", edgecolor="white", linewidth=0.035)
    gdf[gdf[plot_column].notna()].plot(
        ax=ax,
        column=plot_column,
        cmap=spec["cmap"],
        norm=spec["norm"],
        edgecolor="white",
        linewidth=0.025,
    )

    ax.set_xlim(-180, 180)
    ax.set_ylim(-60, 85)
    ax.set_title(spec["title"], pad=7, fontsize=12)
    ax.set_xlabel("Longitude", labelpad=5)
    ax.set_ylabel("Latitude", labelpad=5)
    ax.set_xticks([-150, -100, -50, 0, 50, 100, 150])
    ax.set_yticks([-60, -40, -20, 0, 20, 40, 60, 80])
    ax.tick_params(axis="both", which="major", direction="out", length=6, width=0.9, colors="#111111", top=False, right=False)
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.95)
        spine.set_color("#111111")

    sm = ScalarMappable(cmap=spec["cmap"], norm=spec["norm"])
    sm.set_array([])
    cax = legend_ax.inset_axes([0.31, 0.18, 0.43, 0.22])
    cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")
    cbar.set_label(spec["legend_label"], labelpad=2, fontsize=8.5)
    cbar.set_ticks(spec["ticks"])
    cbar.set_ticklabels(spec["ticklabels"])
    cbar.ax.tick_params(length=4, width=0.8, labelsize=8)
    cbar.outline.set_linewidth(0.8)

    legend_ax.legend(
        handles=[Patch(facecolor="#d9d9d9", edgecolor="white", label="No data")],
        loc="center left",
        bbox_to_anchor=(0.08, 0.30),
        frameon=False,
        handlelength=1.0,
        handleheight=0.8,
        borderaxespad=0,
    )

    fig.savefig(MAP_DIR / spec["filename"], bbox_inches="tight", dpi=300)
    plt.close(fig)

for col in map_specs:
    plot_world_map(col)


## 6. Distribution and Regional Structure Figures

Generate single statistical figures: regional distributions, corridor composition, regional mean ratio, population-weighted summary heatmap, and breadth-depth relationship.

In [ ]:

stat_specs = {
    "EC2015_pc": {
        "title": "Electricity consumption distribution",
        "ylabel": "log10(kWh/cap/year)",
        "plot_col": "log10_EC2015_pc",
        "filename": "Distribution_electricity_consumption_by_region.png",
        "floor_line": np.log10(FLOOR_THRESHOLD),
        "ceiling_line": np.log10(UPPER_THRESHOLD),
        "line_labels": [(np.log10(FLOOR_THRESHOLD), "300"), (np.log10(UPPER_THRESHOLD), "5,000")],
    },
    "sufficiency_ratio": {
        "title": "Sufficiency ratio distribution",
        "ylabel": "log10(ratio)",
        "plot_col": "log10_sufficiency_ratio",
        "filename": "Distribution_sufficiency_ratio_by_region.png",
        "line_labels": [(0, "ratio = 1")],
    },
    "sufficiency_gap": {
        "title": "Sufficiency gap distribution",
        "ylabel": "Population share",
        "plot_col": "sufficiency_gap",
        "filename": "Distribution_sufficiency_gap_by_region.png",
        "ylim": (-0.03, 1.03),
        "line_labels": [],
    },
    "sufficiency_deficit": {
        "title": "Sufficiency deficit distribution",
        "ylabel": "Index",
        "plot_col": "sufficiency_deficit",
        "filename": "Distribution_sufficiency_deficit_by_region.png",
        "ylim": (-0.03, 1.03),
        "line_labels": [],
    },
    "sufficiency_excess": {
        "title": "Sufficiency excess distribution",
        "ylabel": "Population share",
        "plot_col": "sufficiency_excess",
        "filename": "Distribution_sufficiency_excess_by_region.png",
        "ylim": (-0.03, 1.03),
        "line_labels": [],
    },
}

plot_df = df[df["analytical_region"].notna()].copy()
plot_df["log10_EC2015_pc"] = np.nan
mask_ec = plot_df["EC2015_pc"] > 0
plot_df.loc[mask_ec, "log10_EC2015_pc"] = np.log10(plot_df.loc[mask_ec, "EC2015_pc"])
plot_df["log10_sufficiency_ratio"] = np.nan
mask_ratio = plot_df["sufficiency_ratio"] > 0
plot_df.loc[mask_ratio, "log10_sufficiency_ratio"] = np.log10(plot_df.loc[mask_ratio, "sufficiency_ratio"])

for raw_col, spec in stat_specs.items():
    col = spec["plot_col"]
    sub = plot_df[["analytical_region", col]].dropna().copy()
    fig, ax = plt.subplots(figsize=(10.5, 4.8), constrained_layout=True)
    sns.violinplot(
        data=sub,
        x="analytical_region",
        y=col,
        order=REGION_ORDER,
        hue="analytical_region",
        palette=REGION_COLORS,
        legend=False,
        inner=None,
        cut=0,
        linewidth=0,
        alpha=0.72,
        ax=ax,
    )
    sns.boxplot(
        data=sub,
        x="analytical_region",
        y=col,
        order=REGION_ORDER,
        width=0.16,
        showfliers=False,
        boxprops={"facecolor": "white", "edgecolor": "#333333", "linewidth": 0.8, "alpha": 0.72},
        medianprops={"color": "#111111", "linewidth": 1.1},
        whiskerprops={"color": "#333333", "linewidth": 0.8},
        capprops={"color": "#333333", "linewidth": 0.8},
        ax=ax,
    )
    for yval, label in spec.get("line_labels", []):
        ax.axhline(yval, color="#333333", linewidth=0.9, linestyle="--")
        ax.text(0.99, yval, label, transform=ax.get_yaxis_transform(), ha="right", va="bottom", fontsize=8, color="#333333")
    if "ylim" in spec:
        ax.set_ylim(*spec["ylim"])
    ax.set_title(spec["title"], pad=7)
    ax.set_ylabel(spec["ylabel"])
    ax.set_xlabel("")
    ax.set_xticklabels(SHORT_REGION_LABELS, rotation=0)
    ax.tick_params(axis="both", which="major", direction="out", length=5, width=0.8, colors="#111111", top=False, right=False)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.85)
        spine.set_color("#222222")
    fig.savefig(STAT_DIR / spec["filename"], bbox_inches="tight", dpi=300)
    plt.close(fig)


In [ ]:
# Regional corridor composition from pixel-level population shares.
valid_corridor = df[
    df["analytical_region"].notna()
    & df["population"].notna()
    & (df["population"] > 0)
    & df["sufficiency_gap"].notna()
    & df["sufficiency_excess"].notna()
].copy()
valid_corridor["within_corridor"] = (
    1.0 - valid_corridor["sufficiency_gap"] - valid_corridor["sufficiency_excess"]
).clip(lower=0.0, upper=1.0)

corridor_rows = []
for region in REGION_ORDER:
    sub = valid_corridor[valid_corridor["analytical_region"] == region]
    total_pop = sub["population"].sum()
    row = {"analytical_region": region, "population_valid": float(total_pop)}
    pixel_share_cols = {
        "below_floor": "sufficiency_gap",
        "within_corridor": "within_corridor",
        "above_ceiling": "sufficiency_excess",
    }
    for cls, share_col in pixel_share_cols.items():
        pop = (sub["population"] * sub[share_col]).sum()
        row[f"{cls}_population"] = float(pop)
        row[f"{cls}_share"] = float(pop / total_pop) if total_pop > 0 else np.nan
    corridor_rows.append(row)

corridor = pd.DataFrame(corridor_rows)
corridor.to_csv(TABLE_DIR / "rq1_regional_corridor_composition.csv", index=False)

fig, ax = plt.subplots(figsize=(8.2, 4.9), constrained_layout=True)
y = np.arange(len(REGION_ORDER))
left = np.zeros(len(REGION_ORDER))
corridor_colors = {
    "below_floor_share": REFERENCE_COLORS["red"],
    "within_corridor_share": REFERENCE_COLORS["light_blue"],
    "above_ceiling_share": REFERENCE_COLORS["purple"],
}
corridor_labels = {
    "below_floor_share": "Below floor",
    "within_corridor_share": "Within corridor",
    "above_ceiling_share": "Above ceiling",
}
for col in ["below_floor_share", "within_corridor_share", "above_ceiling_share"]:
    vals = corridor[col].fillna(0).to_numpy()
    ax.barh(y, vals, left=left, color=corridor_colors[col], edgecolor="white", linewidth=0.8, label=corridor_labels[col])
    left += vals
ax.set_yticks(y)
ax.set_yticklabels(SHORT_REGION_LABELS)
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel("Population share")
ax.set_title("Pixel-level population-weighted corridor composition", pad=10)
ax.grid(axis="x", color="#d8d8d8", linewidth=0.8)
ax.grid(axis="y", visible=False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=3, frameon=False)
fig.savefig(STAT_DIR / "Regional_corridor_composition_by_region.png", bbox_inches="tight", dpi=300)
plt.close(fig)

ratio_summary = region_summary.copy()
ratio_summary["region_short"] = SHORT_REGION_LABELS
fig, ax = plt.subplots(figsize=(8.0, 4.9), constrained_layout=True)
bar_colors = [REGION_COLORS[r] for r in REGION_ORDER]
ax.barh(y, ratio_summary["sufficiency_ratio_pop_weighted_mean"], color=bar_colors, edgecolor="white", linewidth=0.8)
ax.set_yticks(y)
ax.set_yticklabels(SHORT_REGION_LABELS)
ax.invert_yaxis()
ax.set_xscale("log")
ax.axvline(1, color="#333333", linestyle="--", linewidth=0.9)
ax.axvline(UPPER_THRESHOLD / FLOOR_THRESHOLD, color="#666666", linestyle=":", linewidth=0.9)
ax.set_xlabel("Population-weighted mean sufficiency ratio (log scale)")
ax.set_title("Regional mean sufficiency ratio", pad=10)
ax.grid(axis="x", color="#d8d8d8", linewidth=0.8, which="major")
ax.grid(axis="y", color="#eeeeee", linewidth=0.6)
ax.text(1, -0.55, "floor", ha="center", va="bottom", fontsize=8.5, color="#333333")
ax.text(UPPER_THRESHOLD / FLOOR_THRESHOLD, -0.55, "ceiling", ha="center", va="bottom", fontsize=8.5, color="#666666")
fig.savefig(STAT_DIR / "Regional_mean_sufficiency_ratio_by_region.png", bbox_inches="tight", dpi=300)
plt.close(fig)

heat_source = region_summary.set_index("analytical_region").loc[REGION_ORDER]
heat_values = pd.DataFrame({
    "ratio": np.log10(heat_source["sufficiency_ratio_pop_weighted_mean"].clip(lower=1e-12)),
    "gap": heat_source["sufficiency_gap_pop_weighted_mean"],
    "deficit": heat_source["sufficiency_deficit_pop_weighted_mean"],
    "excess": heat_source["sufficiency_excess_pop_weighted_mean"],
}, index=REGION_ORDER)
annotations = pd.DataFrame({
    "ratio": heat_source["sufficiency_ratio_pop_weighted_mean"].map(lambda v: f"{v:.2f}" if pd.notna(v) else ""),
    "gap": heat_source["sufficiency_gap_pop_weighted_mean"].map(lambda v: f"{v:.2f}" if pd.notna(v) else ""),
    "deficit": heat_source["sufficiency_deficit_pop_weighted_mean"].map(lambda v: f"{v:.2f}" if pd.notna(v) else ""),
    "excess": heat_source["sufficiency_excess_pop_weighted_mean"].map(lambda v: f"{v:.2f}" if pd.notna(v) else ""),
}, index=REGION_ORDER)
heat_table = heat_values.copy()
heat_table.insert(0, "analytical_region", REGION_ORDER)
heat_table.to_csv(TABLE_DIR / "rq1_regional_population_weighted_summary_heatmap_values.csv", index=False)

fig, ax = plt.subplots(figsize=(8.0, 5.1), constrained_layout=True)
heat_cmap = LinearSegmentedColormap.from_list("summary_scale", ["#f7fbff", REFERENCE_COLORS["light_blue"], REFERENCE_COLORS["purple"]])
sns.heatmap(
    heat_values,
    ax=ax,
    cmap=heat_cmap,
    annot=annotations,
    fmt="",
    linewidths=0.8,
    linecolor="white",
    cbar_kws={"label": "Scaled intensity", "shrink": 0.86},
    vmin=0,
    vmax=max(1.7, np.nanmax(heat_values.to_numpy())),
)
ax.set_yticklabels(SHORT_REGION_LABELS, rotation=0)
ax.set_xticklabels(["ratio", "gap", "deficit", "excess"], rotation=0)
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("Regional population-weighted summary", pad=10)
fig.savefig(STAT_DIR / "Regional_population_weighted_summary_heatmap.png", bbox_inches="tight", dpi=300)
plt.close(fig)

In [ ]:
# Breadth-depth relationship with bubble size showing below-floor population size.
breadth = region_summary.copy()
breadth["below_floor_population_size"] = breadth["population_sum"] * breadth["sufficiency_gap_pop_weighted_mean"]
breadth["region_short"] = SHORT_REGION_LABELS

fig, ax = plt.subplots(figsize=(8.2, 5.7), constrained_layout=True)
max_size = breadth["below_floor_population_size"].max()
min_marker, max_marker = 180, 1700
breadth["marker_size"] = min_marker + (breadth["below_floor_population_size"] / max_size) * (max_marker - min_marker)

for _, row in breadth.iterrows():
    ax.scatter(
        row["sufficiency_gap_pop_weighted_mean"],
        row["sufficiency_deficit_pop_weighted_mean"],
        s=row["marker_size"],
        color=REGION_COLORS[row["analytical_region"]],
        edgecolor="white",
        linewidth=1.0,
        alpha=0.9,
        zorder=3,
    )

label_offsets = {
    "Sub-Saharan Africa": (-0.095, 0.035, "right"),
    "Northern Africa and Western Asia": (0.020, 0.050, "left"),
    "South Asia": (0.025, 0.020, "left"),
    "Eastern and South-eastern Asia": (-0.055, -0.060, "right"),
    "Europe": (0.018, 0.022, "left"),
    "Latin America and the Caribbean": (0.024, -0.055, "left"),
    "Northern America and Oceania": (0.034, 0.035, "left"),
}
for _, row in breadth.iterrows():
    dx, dy, ha = label_offsets[row["analytical_region"]]
    ax.annotate(
        row["region_short"].replace("\n", " "),
        xy=(row["sufficiency_gap_pop_weighted_mean"], row["sufficiency_deficit_pop_weighted_mean"]),
        xytext=(row["sufficiency_gap_pop_weighted_mean"] + dx, row["sufficiency_deficit_pop_weighted_mean"] + dy),
        ha=ha,
        va="center",
        fontsize=10.2,
        color="#333333",
        arrowprops=dict(arrowstyle="-", color="#777777", lw=0.8, shrinkA=5, shrinkB=5),
        zorder=4,
    )

ax.set_xlim(-0.03, 0.96)
ax.set_ylim(-0.03, 0.93)
ax.set_xlabel("Below-floor population share", fontsize=12.5)
ax.set_ylabel("Mean normalized deficit", fontsize=12.5)
ax.set_title("Breadth-depth relationship", pad=8, fontsize=15)
ax.tick_params(axis="both", labelsize=11)
ax.grid(True, color="#e9e9e9", linewidth=0.8)

legend_sizes = [1e8, 5e8, 1e9]
legend_handles = []
for size in legend_sizes:
    marker_size = min_marker + (size / max_size) * (max_marker - min_marker)
    legend_handles.append(plt.scatter([], [], s=marker_size, color="#bdbdbd", edgecolor="white", linewidth=0.8, alpha=0.9))
ax.legend(
    legend_handles,
    ["100 million", "500 million", "1 billion"],
    title="Below-floor\npopulation size",
    loc="lower right",
    frameon=True,
    facecolor="white",
    edgecolor="#d0d0d0",
    fontsize=11,
    title_fontsize=12,
    borderpad=0.9,
    labelspacing=1.2,
)

fig.savefig(STAT_DIR / "Breadth_depth_relationship_by_region.png", bbox_inches="tight", dpi=300)
plt.close(fig)

breadth[["analytical_region", "population_sum", "below_floor_population_size", "sufficiency_gap_pop_weighted_mean", "sufficiency_deficit_pop_weighted_mean"]].to_csv(
    TABLE_DIR / "rq1_breadth_depth_relationship_by_region.csv", index=False
)

## 7. Sufficiency Gap Inequality

Compute macro inequality of `sufficiency_gap` using GDL observations and population weights.

- Weighted Gini is based on the population-weighted Lorenz curve.
- Theil T uses `sum p_i * (x_i / mu) * log(x_i / mu)`, with zero terms handled by limit.
- Atkinson uses epsilon = 0.5 as the primary value; epsilon = 0.25 and 0.75 are saved for sensitivity.
- Gap Palma is the total gap-burden share among the highest-gap 10% of population divided by the share among the lowest-gap 40%.

In [ ]:
analysis_df = df[
    df["population"].notna()
    & (df["population"] > 0)
    & df["sufficiency_gap"].notna()
].copy()
analysis_df["gap_burden"] = analysis_df["population"] * analysis_df["sufficiency_gap"]

input_diagnostics = pd.DataFrame([{
    "input_rows": len(df),
    "valid_rows_for_inequality": len(analysis_df),
    "excluded_rows": len(df) - len(analysis_df),
    "valid_population": analysis_df["population"].sum(),
    "total_gap_burden": analysis_df["gap_burden"].sum(),
    "population_weighted_mean_gap": np.average(analysis_df["sufficiency_gap"], weights=analysis_df["population"]),
}])
input_diagnostics.to_csv(PROCESS_DIR / "rq1_sufficiency_gap_inequality_input_diagnostics.csv", index=False)
input_diagnostics

In [ ]:

def clean_xw(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    mask = np.isfinite(x) & np.isfinite(w) & (w > 0) & (x >= 0)
    return x[mask], w[mask]

def weighted_gini(x, w):
    x, w = clean_xw(x, w)
    if len(x) == 0:
        return np.nan
    total_weight = w.sum()
    total_burden = np.sum(w * x)
    if total_weight <= 0 or total_burden <= 0:
        return np.nan
    order = np.argsort(x, kind="mergesort")
    x, w = x[order], w[order]
    cum_pop = np.cumsum(w) / total_weight
    cum_burden = np.cumsum(w * x) / total_burden
    p = np.r_[0, cum_pop]
    l = np.r_[0, cum_burden]
    area_under_lorenz = np.sum((l[1:] + l[:-1]) * (p[1:] - p[:-1]) / 2)
    return float(1 - 2 * area_under_lorenz)

def weighted_theil_t(x, w):
    x, w = clean_xw(x, w)
    if len(x) == 0:
        return np.nan
    p = w / w.sum()
    mu = np.sum(p * x)
    if mu <= 0:
        return np.nan
    ratio = x / mu
    terms = np.zeros_like(ratio)
    positive = ratio > 0
    terms[positive] = p[positive] * ratio[positive] * np.log(ratio[positive])
    return float(np.sum(terms))

def weighted_atkinson(x, w, epsilon=0.5):
    x, w = clean_xw(x, w)
    if len(x) == 0:
        return np.nan
    p = w / w.sum()
    mu = np.sum(p * x)
    if mu <= 0:
        return np.nan
    ratio = x / mu
    if np.isclose(epsilon, 1.0):
        if np.any(ratio <= 0):
            return np.nan
        equally_distributed_equivalent = np.exp(np.sum(p * np.log(ratio)))
    else:
        power = 1 - epsilon
        if power < 0 and np.any(ratio <= 0):
            return np.nan
        equally_distributed_equivalent = np.sum(p * np.power(ratio, power)) ** (1 / power)
    return float(1 - equally_distributed_equivalent)

def burden_share_between_population_quantiles(x, w, q_low, q_high):
    x, w = clean_xw(x, w)
    if len(x) == 0 or not (0 <= q_low < q_high <= 1):
        return np.nan
    total_w = w.sum()
    total_burden = np.sum(w * x)
    if total_w <= 0 or total_burden <= 0:
        return np.nan
    order = np.argsort(x, kind="mergesort")
    x, w = x[order], w[order]
    start = np.r_[0, np.cumsum(w)[:-1]] / total_w
    end = np.cumsum(w) / total_w
    overlap = np.maximum(0, np.minimum(end, q_high) - np.maximum(start, q_low))
    included_weight = overlap * total_w
    return float(np.sum(included_weight * x) / total_burden)

def gap_palma(x, w):
    bottom40 = burden_share_between_population_quantiles(x, w, 0.0, 0.4)
    top10 = burden_share_between_population_quantiles(x, w, 0.9, 1.0)
    if pd.isna(bottom40) or pd.isna(top10):
        return np.nan, top10, bottom40, "undefined_no_burden"
    if np.isclose(bottom40, 0):
        if top10 > 0:
            return np.nan, top10, bottom40, "undefined_bottom40_zero"
        return np.nan, top10, bottom40, "undefined_zero_top_and_bottom"
    return float(top10 / bottom40), top10, bottom40, "ok"

def population_weighted_quantile(x, w, q):
    x, w = clean_xw(x, w)
    if len(x) == 0:
        return np.nan
    order = np.argsort(x, kind="mergesort")
    x, w = x[order], w[order]
    cum = np.cumsum(w) / w.sum()
    return float(np.interp(q, cum, x))

def inequality_summary(sub, name, scope):
    x = sub["sufficiency_gap"].to_numpy()
    w = sub["population"].to_numpy()
    x_clean, w_clean = clean_xw(x, w)
    palma, top10, bottom40, palma_status = gap_palma(x_clean, w_clean)
    total_pop = float(np.sum(w_clean)) if len(w_clean) else np.nan
    total_burden = float(np.sum(w_clean * x_clean)) if len(w_clean) else np.nan
    mean_gap = float(total_burden / total_pop) if total_pop and total_pop > 0 else np.nan
    return {
        "scope": scope,
        "group": name,
        "n_gdl_valid": int(len(x_clean)),
        "valid_population": total_pop,
        "total_gap_burden": total_burden,
        "mean_gap_population_weighted": mean_gap,
        "gap_p10_population_weighted": population_weighted_quantile(x_clean, w_clean, 0.10),
        "gap_p50_population_weighted": population_weighted_quantile(x_clean, w_clean, 0.50),
        "gap_p90_population_weighted": population_weighted_quantile(x_clean, w_clean, 0.90),
        "gini_population_weighted": weighted_gini(x_clean, w_clean),
        "theil_t_population_weighted": weighted_theil_t(x_clean, w_clean),
        "atkinson_epsilon_0_25": weighted_atkinson(x_clean, w_clean, 0.25),
        "atkinson_epsilon_0_5": weighted_atkinson(x_clean, w_clean, 0.5),
        "atkinson_epsilon_0_75": weighted_atkinson(x_clean, w_clean, 0.75),
        "palma_gap_ratio": palma,
        "top10_gap_burden_share": top10,
        "bottom40_gap_burden_share": bottom40,
        "palma_status": palma_status,
    }


In [ ]:

global_result = pd.DataFrame([inequality_summary(analysis_df, "Global", "global")])
region_results = pd.DataFrame([
    inequality_summary(analysis_df[analysis_df["analytical_region"] == region], region, "region")
    for region in REGION_ORDER
])
combined_results = pd.concat([global_result, region_results], ignore_index=True)

# User-facing main table uses Atkinson epsilon=0.5 but sensitivity columns are kept.
main_cols = [
    "scope", "group", "n_gdl_valid", "valid_population", "total_gap_burden", "mean_gap_population_weighted",
    "gini_population_weighted", "theil_t_population_weighted", "atkinson_epsilon_0_5", "palma_gap_ratio",
    "top10_gap_burden_share", "bottom40_gap_burden_share", "palma_status",
]

global_result[main_cols].to_csv(TABLE_DIR / "rq1_sufficiency_gap_inequality_global.csv", index=False)
region_results[main_cols].to_csv(TABLE_DIR / "rq1_sufficiency_gap_inequality_by_region.csv", index=False)
combined_results[main_cols].to_csv(TABLE_DIR / "rq1_sufficiency_gap_inequality_global_and_regions.csv", index=False)
combined_results.to_csv(PROCESS_DIR / "rq1_sufficiency_gap_inequality_full_metrics.csv", index=False)

atkinson_sensitivity = combined_results[["scope", "group", "atkinson_epsilon_0_25", "atkinson_epsilon_0_5", "atkinson_epsilon_0_75"]].copy()
atkinson_sensitivity.to_csv(TABLE_DIR / "rq1_sufficiency_gap_atkinson_sensitivity.csv", index=False)

combined_results[main_cols]


In [ ]:

def format_population_millions(x):
    return x / 1e6

plot_results = region_results.copy()
REGION_LABEL_MAP = dict(zip(REGION_ORDER, SHORT_REGION_LABELS))
plot_results["short_label"] = plot_results["group"].map(REGION_LABEL_MAP)
plot_results["color"] = plot_results["group"].map(REGION_COLORS)

def save_metric_bar(metric, title, ylabel, filename, ylim=None, log_scale=False):
    fig, ax = plt.subplots(figsize=(8.2, 4.8), constrained_layout=True)
    values = plot_results[metric].astype(float)
    bars = ax.bar(plot_results["short_label"], values, color=plot_results["color"], edgecolor="white", linewidth=0.8)
    global_value = float(global_result.iloc[0][metric]) if pd.notna(global_result.iloc[0][metric]) else np.nan
    if pd.notna(global_value):
        ax.axhline(global_value, color="#333333", linestyle="--", linewidth=1.1, label=f"Global = {global_value:.3g}")
        ax.legend(loc="upper left", frameon=True, facecolor="white", edgecolor="#d8d8d8")
    if log_scale:
        ax.set_yscale("log")
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.set_title(title, pad=10)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    ax.grid(axis="y", color="#e5e5e5", linewidth=0.8)
    ax.grid(axis="x", visible=False)
    ax.tick_params(axis="x", labelrotation=0)
    for bar, val in zip(bars, values):
        if pd.notna(val) and np.isfinite(val):
            ax.text(bar.get_x() + bar.get_width()/2, val, f"{val:.2f}" if val < 10 else f"{val:.1f}",
                    ha="center", va="bottom", fontsize=8.5, color="#333333")
    fig.savefig(INEQ_DIR / filename, bbox_inches="tight", dpi=300)
    plt.close(fig)

save_metric_bar("gini_population_weighted", "Population-weighted Gini of sufficiency gap", "Gini", "Gini_sufficiency_gap_by_region.png", ylim=(0, 1))
save_metric_bar("theil_t_population_weighted", "Theil T of sufficiency gap", "Theil T", "TheilT_sufficiency_gap_by_region.png")
save_metric_bar("atkinson_epsilon_0_5", "Atkinson index of sufficiency gap", "Atkinson (epsilon = 0.5)", "Atkinson_sufficiency_gap_by_region.png", ylim=(0, 1))
save_metric_bar("palma_gap_ratio", "Gap Palma ratio", "Top 10% gap share / bottom 40% gap share", "Palma_sufficiency_gap_by_region.png", ylim=(0.1, 100), log_scale=True)

heat = combined_results.set_index("group")[[
    "gini_population_weighted", "theil_t_population_weighted", "atkinson_epsilon_0_5", "palma_gap_ratio"
]].copy()
heat.index = ["Global"] + [REGION_LABEL_MAP[r].replace("\n", " ") for r in REGION_ORDER]
# Column-wise min-max scaling for visual comparison across different metric scales.
heat_scaled = heat.copy()
for col in heat_scaled.columns:
    vals = heat_scaled[col].replace([np.inf, -np.inf], np.nan)
    mn, mx = vals.min(skipna=True), vals.max(skipna=True)
    heat_scaled[col] = (vals - mn) / (mx - mn) if pd.notna(mx) and mx > mn else np.nan

fig, ax = plt.subplots(figsize=(7.6, 5.2), constrained_layout=True)
sns.heatmap(
    heat_scaled,
    annot=heat.round(3).astype(str),
    fmt="",
    cmap=mpl.colors.LinearSegmentedColormap.from_list("ineq_heat", ["#f7fbff", REFERENCE_COLORS["light_blue"], REFERENCE_COLORS["purple"]]),
    linewidths=0.8,
    linecolor="white",
    cbar_kws={"label": "Column-scaled intensity", "shrink": 0.85},
    ax=ax,
)
ax.set_title("Sufficiency gap inequality summary", pad=10)
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_xticklabels(["Gini", "Theil T", "Atkinson", "Palma"], rotation=0)
fig.savefig(INEQ_DIR / "Sufficiency_gap_inequality_summary_heatmap.png", bbox_inches="tight", dpi=300)
plt.close(fig)


In [ ]:

method_metadata = {
    "analysis": "RQ1 sufficiency gap macro inequality",
    "input_data": str(DATA_PATH),
    "unit": "GDL region",
    "weight": "population",
    "variable": "sufficiency_gap",
    "valid_sample_rule": "population > 0 and sufficiency_gap is not NA",
    "interpretation": "Measures inequality in the distribution of sufficiency-gap burden across population-weighted GDL regions. sufficiency_gap is a burden variable, not a welfare variable.",
    "gini": "Weighted Lorenz-curve Gini using population weights.",
    "theil_t": "Sum p_i * (x_i / mu) * ln(x_i / mu), with zero terms treated by the limit 0 ln 0 = 0.",
    "atkinson": "Primary epsilon = 0.5; epsilon 0.25 and 0.75 also saved as sensitivity checks. Epsilon below 1 avoids undefined behavior at x_i = 0.",
    "palma": "Gap-burden Palma: share of total gap burden among the highest-gap 10% of population divided by the share among the lowest-gap 40% of population; observations crossing quantile cutoffs are split proportionally by population weight.",
    "outputs": {
        "tables": sorted([p.name for p in TABLE_DIR.glob("*.csv")]),
        "figures": sorted([p.name for p in INEQ_DIR.glob("*.png")]),
        "process_files": [
            "rq1_sufficiency_gap_inequality_input_diagnostics.csv",
            "rq1_sufficiency_gap_inequality_full_metrics.csv",
            "rq1_sufficiency_gap_inequality_method_metadata.json",
        ],
    },
}
(PROCESS_DIR / "rq1_sufficiency_gap_inequality_method_metadata.json").write_text(json.dumps(method_metadata, indent=2), encoding="utf-8")
method_metadata


## 8. Output Manifest

Save a compact manifest so the generated tables, figures, and method choices can be checked without reopening the notebook.

In [ ]:

manifest = {
    "input_data": str(DATA_PATH),
    "input_shapefile": str(SHP_PATH),
    "output_directory": str(RESULTS_DIR),
    "map_pngs": sorted([p.name for p in MAP_DIR.glob("*.png")]),
    "statistic_pngs": sorted([p.name for p in STAT_DIR.glob("*.png")]),
    "inequality_pngs": sorted([p.name for p in INEQ_DIR.glob("*.png")]),
    "tables": sorted([p.name for p in TABLE_DIR.glob("*.csv")]),
    "region_palette": REGION_COLORS,
    "floor_threshold_kwh_per_capita_year": FLOOR_THRESHOLD,
    "upper_threshold_kwh_per_capita_year": UPPER_THRESHOLD,
}
(PROCESS_DIR / "rq1_publish_ready_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
manifest
